# Overfit Test: Can the model learn at all?

**Goal:** Train on a tiny dataset (1 AOI) and validate/test on the **same** data.  
If the model can't reach near-perfect PSNR on data it's seen, something is fundamentally broken.

**Key changes vs main notebook:**
- Single AOI, same tiles for train/val/test
- No augmentation (maximize memorization)
- `* 0.1` residual scaling **removed**
- Bilinear skip (not bicubic)
- Higher LR (1e-3)
- Frequent validation (every 500 iters)

In [ ]:
# ============================================================
# Cell 0 — Setup
# ============================================================
!git clone https://github.com/XPixelGroup/BasicSR.git 2>/dev/null || echo 'BasicSR already cloned'
%cd /content/BasicSR
!pip install -e . -q
!pip install rasterio -q

import sys, os
sys.path.insert(0, '/content/BasicSR')

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# Cell 1 — Data prep: single AOI, same data everywhere
# ============================================================
# Pick ONE AOI with tiles already on Drive.
# All tiles go into train, val, AND test (identical).

from pathlib import Path
import numpy as np
import rasterio
import re
from tqdm import tqdm

# ================= CONFIG — EDIT THESE =================
# Point to ONE specific AOI's pair folder
PAIR_DIR = Path("/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-17/aoi_XXXX/pair_YYYY")  # ← EDIT

# Where your LR and GT tiles live within that pair
LR_DIR = PAIR_DIR / "tiles"          # contains *_emit_b32.tif
GT_DIR = PAIR_DIR / "SFIM_Bilinear"  # contains *_SFIM_fused.tif

# Max tiles to use (keep small for fast overfitting)
MAX_TILES = 10

OUT_DIR = Path("/content/overfit_data")
EMIT_SCALE = 10000.0
NODATA_U16 = 65535
# =======================================================


def _read_emit_b32(path):
    with rasterio.open(path) as ds:
        arr = ds.read().astype(np.float32)
    mask = arr == NODATA_U16
    arr /= EMIT_SCALE
    arr[mask] = 0.0
    return arr


def _read_gt(path):
    with rasterio.open(path) as ds:
        arr = ds.read().astype(np.float32)
    arr /= EMIT_SCALE
    return np.clip(np.nan_to_num(arr, nan=0.0), 0.0, None)


# Find and pair tiles
lr_files = sorted(LR_DIR.glob("*_emit_b32.tif"))
gt_by_id = {}
for p in GT_DIR.glob("*_SFIM_fused.tif"):
    m = re.search(r'tile(\d+)', p.stem)
    if m:
        gt_by_id[int(m.group(1))] = p

pairs = []
for lr_path in lr_files:
    m = re.search(r'tile(\d+)', lr_path.stem)
    if not m:
        continue
    tid = int(m.group(1))
    if tid in gt_by_id:
        pairs.append((lr_path, gt_by_id[tid], tid))

pairs = pairs[:MAX_TILES]
print(f"Using {len(pairs)} tile pairs for overfit test")

# Save SAME tiles to train, val, and test
for split in ['train', 'val', 'test']:
    (OUT_DIR / split / 'lq').mkdir(parents=True, exist_ok=True)
    (OUT_DIR / split / 'gt').mkdir(parents=True, exist_ok=True)

for lr_path, gt_path, tid in tqdm(pairs, desc="Preparing"):
    lr = _read_emit_b32(lr_path)
    gt = _read_gt(gt_path)
    stem = f"tile{tid:03d}"

    for split in ['train', 'val', 'test']:
        np.save(OUT_DIR / split / 'lq' / f"{stem}.npy", lr)
        np.save(OUT_DIR / split / 'gt' / f"{stem}.npy", gt)

# Quick sanity check
lr0 = np.load(OUT_DIR / "train" / "lq" / f"tile{pairs[0][2]:03d}.npy")
gt0 = np.load(OUT_DIR / "train" / "gt" / f"tile{pairs[0][2]:03d}.npy")
print(f"LR: {lr0.shape}, range [{lr0.min():.4f}, {lr0.max():.4f}]")
print(f"GT: {gt0.shape}, range [{gt0.min():.4f}, {gt0.max():.4f}]")
print(f"Scale: {gt0.shape[1]/lr0.shape[1]:.0f}x")

In [ ]:
# ============================================================
# Cell 2 — Dataset (no augmentation for overfit test)
# ============================================================
import torch
from torch.utils.data import Dataset
from basicsr.utils.registry import DATASET_REGISTRY
from pathlib import Path
import numpy as np
import random

if 'PairedNpyDataset' in DATASET_REGISTRY._obj_map:
    del DATASET_REGISTRY._obj_map['PairedNpyDataset']


@DATASET_REGISTRY.register()
class PairedNpyDataset(Dataset):
    def __init__(self, opt):
        super().__init__()
        self.opt = opt
        self.gt_dir = Path(opt['dataroot_gt'])
        self.lq_dir = Path(opt['dataroot_lq'])
        self.scale = opt.get('scale', 6)
        self.gt_size = opt.get('gt_size', 120)

        self.gt_files = sorted(self.gt_dir.glob('*.npy'))
        self.lq_files = sorted(self.lq_dir.glob('*.npy'))
        assert len(self.gt_files) == len(self.lq_files), \
            f"Mismatch: {len(self.gt_files)} GT vs {len(self.lq_files)} LQ"
        assert len(self.gt_files) > 0, f"No .npy files in {self.gt_dir}"

        for g, l in zip(self.gt_files, self.lq_files):
            assert g.stem == l.stem, f"Name mismatch: {g.stem} vs {l.stem}"

        print(f"[PairedNpyDataset] {len(self.gt_files)} pairs, "
              f"scale={self.scale}, gt_size={self.gt_size}")

    def __len__(self):
        return len(self.gt_files)

    def __getitem__(self, idx):
        gt = np.load(self.gt_files[idx]).astype(np.float32)
        lq = np.load(self.lq_files[idx]).astype(np.float32)

        # Random crop only during training
        if self.opt.get('phase') == 'train':
            C, H_gt, W_gt = gt.shape
            gt_size = self.gt_size
            lq_size = gt_size // self.scale

            top_gt  = random.randint(0, max(0, H_gt - gt_size))
            left_gt = random.randint(0, max(0, W_gt - gt_size))
            top_lq  = top_gt  // self.scale
            left_lq = left_gt // self.scale

            gt = gt[:, top_gt:top_gt+gt_size, left_gt:left_gt+gt_size]
            lq = lq[:, top_lq:top_lq+lq_size, left_lq:left_lq+lq_size]

            # NO augmentation — we want pure memorization

        return {
            'lq': torch.from_numpy(lq.copy()).float(),
            'gt': torch.from_numpy(gt.copy()).float(),
            'lq_path': str(self.lq_files[idx]),
            'gt_path': str(self.gt_files[idx]),
        }

print("PairedNpyDataset registered (no augmentation).")

In [ ]:
# ============================================================
# Cell 3 — RRDBNet6x (FIXED: no *0.1, bilinear skip)
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from basicsr.archs.rrdbnet_arch import RRDB
from basicsr.utils.registry import ARCH_REGISTRY

if 'RRDBNet6x' in ARCH_REGISTRY._obj_map:
    del ARCH_REGISTRY._obj_map['RRDBNet6x']


@ARCH_REGISTRY.register()
class RRDBNet6x(nn.Module):
    def __init__(self, num_in_ch, num_out_ch, num_feat=64,
                 num_block=23, num_grow_ch=32):
        super().__init__()
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)

        self.body = nn.ModuleList()
        for _ in range(num_block):
            self.body.append(RRDB(num_feat, num_grow_ch=num_grow_ch))
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)

        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)  # after 2x
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)  # after 3x

        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)

        # Zero-init last conv so initial output = bilinear skip
        nn.init.zeros_(self.conv_last.weight)
        nn.init.zeros_(self.conv_last.bias)

        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        feat = self.conv_first(x)

        body_feat = feat
        for block in self.body:
            body_feat = block(body_feat)
        body_feat = self.conv_body(body_feat)
        feat = feat + body_feat

        # 2x then 3x = 6x
        feat = F.interpolate(feat, scale_factor=2, mode='nearest')
        feat = self.lrelu(self.conv_up1(feat))
        feat = F.interpolate(feat, scale_factor=3, mode='nearest')
        feat = self.lrelu(self.conv_up2(feat))

        out = self.conv_last(self.lrelu(self.conv_hr(feat)))

        # FIXED: bilinear skip, NO * 0.1 scaling
        base = F.interpolate(x, scale_factor=6, mode='bilinear', align_corners=False)
        return out + base


# Sanity check
_net = RRDBNet6x(num_in_ch=32, num_out_ch=32, num_feat=64, num_block=4)
_x = torch.randn(1, 32, 20, 20)
_y = _net(_x)
print(f"Input: {_x.shape} -> Output: {_y.shape}")
assert _y.shape == (1, 32, 120, 120), f"Expected (1,32,120,120), got {_y.shape}"
del _net, _x, _y
print("RRDBNet6x registered (bilinear skip, no 0.1 scaling).")

In [ ]:
# ============================================================
# Cell 4 — Config for overfit test
# ============================================================
import yaml
from pathlib import Path

EXP_NAME   = 'overfit_test'
DATA_ROOT  = '/content/overfit_data'
NUM_BANDS  = 32
SCALE      = 6
GT_SIZE    = 120    # full tile = no random crop variance
BATCH_SIZE = 4
TOTAL_ITER = 5_000  # should be plenty to overfit 10 tiles
LR_RATE    = 1e-3   # aggressive LR for fast overfitting
NUM_BLOCK  = 8      # small model — faster, still enough capacity
NUM_FEAT   = 64

config = {
    'name': EXP_NAME,
    'model_type': 'SRModel',
    'scale': SCALE,
    'num_gpu': 1,
    'manual_seed': 42,

    'datasets': {
        'train': {
            'name': 'overfit_train',
            'type': 'PairedNpyDataset',
            'dataroot_gt': f'{DATA_ROOT}/train/gt',
            'dataroot_lq': f'{DATA_ROOT}/train/lq',
            'gt_size': GT_SIZE,
            'scale': SCALE,
            'use_hflip': False,   # NO augmentation
            'use_rot': False,
            'num_worker_per_gpu': 2,
            'batch_size_per_gpu': BATCH_SIZE,
            'dataset_enlarge_ratio': 100,
        },
        'val': {
            'name': 'overfit_val',
            'type': 'PairedNpyDataset',
            'dataroot_gt': f'{DATA_ROOT}/val/gt',
            'dataroot_lq': f'{DATA_ROOT}/val/lq',
            'gt_size': GT_SIZE,
            'scale': SCALE,
        },
    },

    'network_g': {
        'type': 'RRDBNet6x',
        'num_in_ch': NUM_BANDS,
        'num_out_ch': NUM_BANDS,
        'num_feat': NUM_FEAT,
        'num_block': NUM_BLOCK,
        'num_grow_ch': 32,
    },

    'path': {
        'pretrain_network_g': None,
        'strict_load_g': True,
        'resume_state': None,
    },

    'train': {
        'ema_decay': 0.999,
        'optim_g': {
            'type': 'Adam',
            'lr': LR_RATE,
            'weight_decay': 0,
            'betas': [0.9, 0.99],
        },
        'scheduler': {
            'type': 'MultiStepLR',
            'milestones': [2000, 3500],
            'gamma': 0.5,
        },
        'total_iter': TOTAL_ITER,
        'warmup_iter': -1,
        'pixel_opt': {
            'type': 'L1Loss',
            'loss_weight': 1.0,
            'reduction': 'mean',
        },
    },

    'val': {
        'val_freq': 500,         # validate often to watch convergence
        'save_img': False,
        'metrics': {
            'psnr': {
                'type': 'calculate_psnr',
                'crop_border': SCALE,
                'test_y_channel': False,
            },
        },
    },

    'logger': {
        'print_freq': 50,        # log every 50 iters
        'save_checkpoint_freq': 1000,
        'use_tb_logger': True,
    },

    'dist_params': {
        'backend': 'nccl',
        'port': 29500,
    },
}

config_path = Path('/content/BasicSR/options/train') / f'{EXP_NAME}.yml'
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config: {config_path}")
print(f"  {NUM_BLOCK} blocks, {NUM_FEAT} feats, LR={LR_RATE}")
print(f"  {TOTAL_ITER} iters, val every 500, batch={BATCH_SIZE}")
print(f"  GT crop: {GT_SIZE} -> LR: {GT_SIZE//SCALE}")

In [ ]:
# ============================================================
# Cell 5 — Patch BasicSR's tensor2img for 32-band tensors
# ============================================================
import importlib
import numpy as np
import basicsr.utils.img_util as _iu
import basicsr.models.sr_model as _srm

importlib.reload(_iu)

_orig = _iu.tensor2img

def _patched_tensor2img(tensor, rgb2bgr=True, out_type=np.uint8, min_max=(0, 1)):
    t = tensor[0] if isinstance(tensor, list) else tensor
    if t.ndim >= 3 and t.shape[-3] > 4:
        rgb2bgr = False
    return _orig(tensor, rgb2bgr=rgb2bgr, out_type=out_type, min_max=min_max)

_iu.tensor2img = _patched_tensor2img
_srm.tensor2img = _patched_tensor2img
print("Patched tensor2img for 32-band support.")

In [ ]:
# ============================================================
# Cell 6 — Train!
# ============================================================
import sys
sys.path.insert(0, '/content/BasicSR')

from basicsr.train import train_pipeline
import basicsr.train as train_module

sys.argv = ['train.py', '-opt', str(config_path)]

try:
    train_pipeline('/content/BasicSR')
except Exception as e:
    print(f"Error: {e}")
    raise

In [ ]:
# ============================================================
# Cell 7 — Quick visual check: did it overfit?
# ============================================================
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Find latest checkpoint
import re
ckpt_dir = Path(f'/content/BasicSR/experiments/{EXP_NAME}/models')
ckpts = sorted(ckpt_dir.glob('net_g_*.pth'))
if not ckpts:
    print(f"No checkpoints in {ckpt_dir}")
else:
    ckpt = ckpts[-1]
    print(f"Loading: {ckpt}")

    net = RRDBNet6x(num_in_ch=32, num_out_ch=32,
                    num_feat=NUM_FEAT, num_block=NUM_BLOCK)
    state = torch.load(ckpt, map_location='cpu')
    key = 'params_ema' if 'params_ema' in state else 'params' if 'params' in state else None
    net.load_state_dict(state[key] if key else state)
    net = net.cuda().eval()

    # Load a training tile (should be memorized)
    DATA = Path("/content/overfit_data")
    lq = np.load(sorted((DATA / "train" / "lq").glob("*.npy"))[0])
    gt = np.load(sorted((DATA / "train" / "gt").glob("*.npy"))[0])

    with torch.no_grad():
        sr = net(torch.from_numpy(lq[None]).float().cuda()).squeeze(0).cpu().numpy()

    bilinear = F.interpolate(
        torch.from_numpy(lq[None]).float(),
        scale_factor=6, mode='bilinear', align_corners=False
    ).squeeze(0).numpy()

    # RGB from bands 10, 6, 2
    def to_rgb(cube, bands=(10, 6, 2)):
        rgb = np.stack([cube[b] for b in bands], axis=-1)
        lo, hi = np.percentile(rgb[rgb > 0], [2, 98]) if (rgb > 0).any() else (0, 1)
        return np.clip((rgb - lo) / (hi - lo + 1e-8), 0, 1)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, img, title in zip(axes,
        [to_rgb(lq), to_rgb(bilinear), to_rgb(sr), to_rgb(gt)],
        [f'LR {lq.shape[1]}x{lq.shape[2]}',
         f'Bilinear {bilinear.shape[1]}x{bilinear.shape[2]}',
         f'SR {sr.shape[1]}x{sr.shape[2]}',
         f'GT {gt.shape[1]}x{gt.shape[2]}']):
        ax.imshow(img); ax.set_title(title); ax.axis('off')
    plt.suptitle('Overfit test — training tile (should match GT closely)')
    plt.tight_layout(); plt.show()

    # Per-band PSNR
    cb = 6
    sc = sr[:, cb:-cb, cb:-cb]
    gc = gt[:, cb:-cb, cb:-cb]
    bc = bilinear[:, cb:-cb, cb:-cb]
    mse_sr  = np.mean((sc - gc) ** 2)
    mse_bil = np.mean((bc - gc) ** 2)
    psnr_sr  = 10 * np.log10(1.0 / max(mse_sr, 1e-10))
    psnr_bil = 10 * np.log10(1.0 / max(mse_bil, 1e-10))

    print(f"\nPSNR on training tile:")
    print(f"  SR:       {psnr_sr:.2f} dB")
    print(f"  Bilinear: {psnr_bil:.2f} dB")
    print(f"  Gain:     {psnr_sr - psnr_bil:+.2f} dB")
    print(f"\nIf SR PSNR is NOT significantly above bilinear, something is wrong.")
    print(f"If SR PSNR > 45 dB, the model can learn. Proceed to full training.")